In [ ]:
%%writefile histogram_cuda.cu
#include <cuda_runtime.h>
#include <iostream>
#include <vector>
#include <random>
#include <cstdlib>
#include <cstdint>

constexpr int NUM_BINS = 256;
constexpr int THREADS_PER_BLOCK = 256;

#define CUDA_CHECK(call) \
    do { \
        cudaError_t err__ = (call); \
        if (err__ != cudaSuccess) { \
            std::cerr << "CUDA error at " << __FILE__ << ":" << __LINE__ \
                      << " - " << cudaGetErrorString(err__) << std::endl; \
            std::exit(EXIT_FAILURE); \
        } \
    } while (0)

__global__ void histogramKernel(const unsigned char* input, unsigned int* histogram, std::size_t n) {
    const std::size_t i = static_cast<std::size_t>(blockIdx.x) * blockDim.x + threadIdx.x;
    if (i < n) {
        const unsigned int bin = static_cast<unsigned int>(input[i]);
        atomicAdd(&histogram[bin], 1U);
    }
}

void histogramCPU(const std::vector<unsigned char>& input, std::vector<unsigned int>& histogram) {
    std::fill(histogram.begin(), histogram.end(), 0U);
    for (unsigned char value : input) {
        ++histogram[static_cast<unsigned int>(value)];
    }
}

int main(int argc, char** argv) {
    std::size_t n = 1U << 20;

    if (argc > 1) {
        char* end = nullptr;
        const unsigned long long parsed = std::strtoull(argv[1], &end, 10);
        if (end == argv[1] || *end != '\0' || parsed == 0ULL) {
            std::cerr << "Usage: " << argv[0] << " [positive_input_size]" << std::endl;
            return EXIT_FAILURE;
        }
        n = static_cast<std::size_t>(parsed);
    }

    std::vector<unsigned char> input(n);
    std::mt19937 generator(12345);
    std::uniform_int_distribution<int> distribution(0, NUM_BINS - 1);

    for (std::size_t i = 0; i < n; ++i) {
        input[i] = static_cast<unsigned char>(distribution(generator));
    }

    std::vector<unsigned int> cpuHistogram(NUM_BINS, 0U);
    std::vector<unsigned int> gpuHistogram(NUM_BINS, 0U);

    histogramCPU(input, cpuHistogram);

    unsigned char* d_input = nullptr;
    unsigned int* d_histogram = nullptr;

    CUDA_CHECK(cudaMalloc(reinterpret_cast<void**>(&d_input), n * sizeof(unsigned char)));
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void**>(&d_histogram), NUM_BINS * sizeof(unsigned int)));

    CUDA_CHECK(cudaMemcpy(d_input, input.data(), n * sizeof(unsigned char), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemset(d_histogram, 0, NUM_BINS * sizeof(unsigned int)));

    const std::size_t blocks = (n + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;

    histogramKernel<<<static_cast<unsigned int>(blocks), THREADS_PER_BLOCK>>>(d_input, d_histogram, n);

    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    CUDA_CHECK(cudaMemcpy(gpuHistogram.data(), d_histogram, NUM_BINS * sizeof(unsigned int), cudaMemcpyDeviceToHost));

    bool matches = true;
    for (int bin = 0; bin < NUM_BINS; ++bin) {
        if (cpuHistogram[bin] != gpuHistogram[bin]) {
            std::cerr << "Mismatch at bin " << bin << ": CPU=" << cpuHistogram[bin]
                      << ", GPU=" << gpuHistogram[bin] << std::endl;
            matches = false;
        }
    }

    std::uint64_t total = 0;
    for (unsigned int count : gpuHistogram) {
        total += count;
    }

    std::cout << "Input elements: " << n << std::endl;
    std::cout << "Histogram bins: " << NUM_BINS << std::endl;
    std::cout << "Sum of GPU bins: " << total << std::endl;
    std::cout << "CPU/GPU histogram comparison: " << (matches ? "PASS" : "FAIL") << std::endl;

    CUDA_CHECK(cudaFree(d_histogram));
    CUDA_CHECK(cudaFree(d_input));

    if (!matches || total != n) {
        return EXIT_FAILURE;
    }
    return EXIT_SUCCESS;
}

Overwriting histogram_cuda.cu


In [3]:
!nvcc -O2 histogram_cuda.cu -o histogram_cuda

In [4]:
!./histogram_cuda

Input elements: 1048576
Histogram bins: 256
Sum of GPU bins: 1048576
CPU/GPU histogram comparison: PASS
